In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score, log_loss


In [ ]:
#Loading data
try:
    train = pd.read_csv('TrainData.csv')
    test = pd.read_csv('TestData.csv')
    sub = pd.DataFrame({
        'LeadID': test['LeadID'],
        'VehicleSold': 0
    })
    print("Data loaded.")
except FileNotFoundError:
    print("Missing CSV files, Please make sure they are in the same folder")


In [ ]:
#Performing feature engineering
def make_features(df):
    df['DTLeadCreated'] = pd.to_datetime(df['DTLeadCreated'], errors='coerce')
    df['DTLeadAllocated'] = pd.to_datetime(df['DTLeadAllocated'], errors='coerce')

    df['alloc_min'] = (df['DTLeadAllocated'] - df['DTLeadCreated']).dt.total_seconds() / 60
    df['never_alloc'] = df['DTLeadAllocated'].isna().astype(int)
    df['hour'] = df['DTLeadCreated'].dt.hour
    df['day'] = df['DTLeadCreated'].dt.dayofweek
    df['mon'] = df['DTLeadCreated'].dt.month
    df['week'] = df['DTLeadCreated'].dt.isocalendar().week.astype(int)

    df['cust_count'] = df.groupby('CustomerID')['CustomerID'].transform('count')
    df['combo'] = df['InterestMake'].astype(str) + '_' + df['InterestModel'].astype(str)

    df['alloc_min'] = df['alloc_min'].fillna(-999)
    df['InterestMake'] = df['InterestMake'].fillna('No InterestMake')
    df['InterestModel'] = df['InterestModel'].fillna('No InterestModel')
    df['Domain'] = df['Domain'].fillna('No Domain')
    if 'VehicleSold' in df.columns:
        df['VehicleSold'] = df['VehicleSold'].fillna(0)  #Ensuring no NaN in target

    return df


In [ ]:
#Applying feature engineering
train = make_features(train)
test = make_features(test)
print("Feature engineering complete.")

drop_cols = [
    'LeadID', 'CustomerID', 'DTLeadCreated', 'DTLeadAllocated',
    'OBSFullName', 'OBSEmail', 'VehicleSold',
    'InFinanceProcessSystemApp', 'FinanceApplied', 'FinanceApproved'
]

y = train['VehicleSold'].copy()
feats = [f for f in train.columns if f not in drop_cols]

x = train[feats].copy()
xt = test[feats].copy()


In [ ]:
#Identifing categorical columns before Label Encoding for CatBoost's native handling
original_cat_cols = x.select_dtypes(include='object').columns.tolist()

for col in original_cat_cols: #Changed `cat_cols` to `original_cat_cols` for clarity
    le = LabelEncoder()
    full_col_data = pd.concat([x[col], xt[col]], axis=0).astype(str)
    le.fit(full_col_data)
    x[col] = le.transform(x[col].astype(str))
    xt[col] = le.transform(xt[col].astype(str))

print(f"Final features used: {len(feats)}")


In [ ]:
print("Starting Hyperparameter Tuning with GridSearchCV")

base_cat_model = cb.CatBoostClassifier(
    objective='Logloss',
    eval_metric='PRAUC',
    random_seed=1,
    verbose=0,
    early_stopping_rounds=50,
    scale_pos_weight=(len(y) - y.sum()) / y.sum(),
    cat_features=cat_feature_indices
)

param_grid = {
    'iterations': [500, 1000], #Reduced for faster execution in demo
    'learning_rate': [0.03, 0.05],
    'depth': [4, 6],
    'l2_leaf_reg': [1, 3]

}

kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=1) #Reduced folds for faster tuning in demo

grid_search = GridSearchCV(
    estimator=base_cat_model,
    param_grid=param_grid,
    cv=kf,
    scoring='average_precision',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(x, y)

print("\nHyperparameter tuning complete.")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best PR-AUC score (GridSearchCV): {grid_search.best_score_:.5f}")

#Training the final CatBoost model with the best parameters
print("\nTraining final model with best parameters...")
best_cat_params = grid_search.best_params_
final_cat_model = cb.CatBoostClassifier(
    objective='Logloss',
    eval_metric='PRAUC',
    random_seed=1,
    verbose=100,
    early_stopping_rounds=50,
    scale_pos_weight=(len(y) - y.sum()) / y.sum(),
    cat_features=cat_feature_indices,
    **best_cat_params #Unpacking the best parameters found by GridSearchCV
)

# Re-running the KFold with the best parameters found to get proper OOF predictions
oof_tuned = np.zeros(len(x))
preds_tuned = np.zeros(len(xt))
importances_tuned = []

print("Running KFold with best parameters to get OOF and final predictions")
for fold, (trn_idx, val_idx) in enumerate(kf.split(x, y)):
    print(f"Fold {fold+1}")
    fold_model = cb.CatBoostClassifier(
        objective='Logloss',
        eval_metric='PRAUC',
        random_seed=1,
        verbose=0,
        early_stopping_rounds=50,
        scale_pos_weight=(len(y) - y.sum()) / y.sum(),
        cat_features=cat_feature_indices,
        **best_cat_params
    )
    fold_model.fit(
        x.iloc[trn_idx], y.iloc[trn_idx],
        eval_set=(x.iloc[val_idx], y.iloc[val_idx]),
        verbose=False # Seting to True for verbose output during training
    )

    oof_tuned[val_idx] = fold_model.predict_proba(x.iloc[val_idx])[:, 1]
    preds_tuned += fold_model.predict_proba(xt)[:, 1] / kf.n_splits
    importances_tuned.append(fold_model.get_feature_importance())



In [ ]:
#Evaluating the Tuned Model
score_tuned = average_precision_score(y, oof_tuned)
print("\nOOF PR-AUC (Tuned CatBoost):", round(score_tuned, 5))
logloss_tuned = log_loss(y, oof_tuned)
print("OOF Log Loss (Tuned CatBoost):", round(logloss_tuned, 5))


In [ ]:
sub['VehicleSold'] = preds_tuned
sub.to_csv('submission_catboost_tuned.csv', index=False)
print("Submission saved to 'submission_catboost_tuned.csv'")


In [ ]:
#Plotting feature importances (from the last fold or a refitted model)

mean_importances = np.mean(importances_tuned, axis=0)

fi = pd.DataFrame({
    'feature': feats,
    'importance': mean_importances
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='importance', y='feature', data=fi.head(15), palette='viridis')
plt.title('Top 15 most important features (Tuned CatBoost)', fontsize=16)
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:
#Allocation Time Analysis (unchanged) using the original train dataframe before encoding
train['alloc_bin'] = pd.cut(train['alloc_min'],
                             bins=[-1000, 0, 30, 60, 1440, float('inf')],
                             labels=['Never Allocated', '< 30 min', '30-60 min', '1-24 hours', '> 24 hours'])

alloc_corr = train.groupby('alloc_bin')['VehicleSold'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=alloc_corr.index, y=alloc_corr.values, palette='plasma')
plt.title('Conversion Rate by Lead Allocation Time', fontsize=16)
plt.ylabel('Vehicle Sold Rate')
plt.xlabel('Allocation Time Bin')
plt.xticks(rotation=45)
plt.show()


In [ ]:
#Lead Source Analysis (unchanged) - Getting the top 15 lead sources by count
top_sources = train['LeadSource'].value_counts().nlargest(15).index
train_top_sources = train[train['LeadSource'].isin(top_sources)]

source_corr = train_top_sources.groupby('LeadSource')['VehicleSold'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x=source_corr.values, y=source_corr.index, orient='h', palette='magma')
plt.title('Conversion Rate for Top 15 Lead Sources', fontsize=16)
plt.xlabel('Vehicle Sold Rate')
plt.ylabel('Lead Source')
plt.tight_layout()
plt.show()
